# 🎙️ NVIDIA Magpie TTS — Multilingual Text-to-Speech

**Model:** [`nvidia/magpie_tts_multilingual_357m`](https://huggingface.co/nvidia/magpie_tts_multilingual_357m)  
**Runtime:** Google Colab (GPU recommended, CPU OK)  
**Languages:** English, Spanish, German, French, Vietnamese, Italian, Mandarin, Hindi, Japanese  
**Speakers:** Sofia (1), Aria (2), Jason (3), Leo (4), John (0)

### Workflow
1. **Step 1** — Install NeMo & dependencies  
2. **Step 2** — Upload your text file  
3. **Step 3** — Configure voice, speed, pitch  
4. **Step 4** — Load the model  
5. **Step 5** — Generate speech  
6. **Step 6** — Download audio  

> ⚙️ **GPU:** `Runtime → Change runtime type → T4 GPU`

---
## 📦 Step 1 — Install Dependencies
Installs NeMo toolkit with TTS support and all required packages.  
> ⏱️ This takes 3–5 minutes on a fresh runtime.

In [ ]:
import subprocess, sys, os, time

print("📦 Installing NVIDIA NeMo and dependencies...")
print("   This takes 3–5 minutes on a fresh Colab runtime.\n")

t0 = time.time()

# Install NeMo with TTS support (official method per HF model card)
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "nemo_toolkit[tts]",
    "kaldialign",
    "soundfile",
    "librosa>=0.10",
])

elapsed = time.time() - t0
print(f"\n✅ Installation complete in {elapsed:.0f}s")

# ── PRE-PATCH FOR PYTHON 3.12 COMPATIBILITY ──
# Colab uses Python 3.12 which removed pkgutil.ImpImporter and FileFinder.find_module.
# Older deps like jieba and hydra-core crash without these monkey-patches.
import sys, pkgutil, importlib.machinery
if sys.version_info >= (3, 12):
    if not hasattr(pkgutil, "ImpImporter"):
        class _DummyImpImporter:
            def __init__(self, path=None): self.path = path
            def find_module(self, fullname, path=None): return None
            def iter_modules(self, prefix=""): return iter([])
        pkgutil.ImpImporter = _DummyImpImporter
    if not hasattr(importlib.machinery.FileFinder, "find_module"):
        def _dummy_find_module(self, fullname, path=None):
            spec = self.find_spec(fullname)
            return spec.loader if spec else None
        importlib.machinery.FileFinder.find_module = _dummy_find_module
# ─────────────────────────────────────────────

# Quick verification
import torch
import numpy as np
print(f"   Python  : {sys.version.split()[0]}")
print(f"   NumPy   : {np.__version__}")
print(f"   PyTorch : {torch.__version__}  CUDA={torch.cuda.is_available()}")

try:
    import nemo
    print(f"   NeMo    : {nemo.__version__}")
except:
    print("   NeMo    : (version check failed, but may still work)")

print("\n✅ All dependencies installed — proceed to Step 2!")

---
## 📤 Step 2 — Upload Your Text File
Upload a `.txt` file with the text you want spoken.

In [ ]:
from google.colab import files as colab_files

print("📁 Please upload a .txt file:")
uploaded = colab_files.upload()

if not uploaded:
    raise ValueError("❌ No file uploaded. Re-run this cell.")

fname = list(uploaded.keys())[0]
INPUT_TEXT = uploaded[fname].decode("utf-8", errors="replace").strip()

if not INPUT_TEXT:
    raise ValueError("❌ File is empty.")

print(f"\n✅ {fname}  |  {len(INPUT_TEXT):,} chars  |  {len(INPUT_TEXT.split()):,} words")
print("\n--- Preview (first 400 chars) ---")
print(INPUT_TEXT[:400] + ("..." if len(INPUT_TEXT) > 400 else ""))
print("---------------------------------")

---
## ⚙️ Step 3 — Configure Voice, Speed, Pitch & Quality
Edit **only this cell** — all parameters are explained inline.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║          🎛️  TTS SETTINGS — EDIT ANYTHING BELOW            ║
# ╚══════════════════════════════════════════════════════════════╝

# ── SPEAKER / VOICE ─────────────────────────────────────────────
# Available speakers: John (0), Sofia (1), Aria (2), Jason (3), Leo (4)
SPEAKER_ID = 1              # int (0–4)
SPEAKER_NAME = {0: "John", 1: "Sofia", 2: "Aria", 3: "Jason", 4: "Leo"}

# ── LANGUAGE ────────────────────────────────────────────────────
# Supported: "english", "spanish", "german", "french", "vietnamese", "italian", "mandarin", "hindi", "japanese"
LANGUAGE = "hindi"             # str — set to match your input text

# ── TEXT NORMALIZATION ──────────────────────────────────────────
# Supported for: en, es, de, fr, it, zh. Set False for other languages.
APPLY_TEXT_NORM = False     # bool

# ── SPEAKING RATE / SPEED ───────────────────────────────────────
# 1.0 = natural  |  0.8 = 20% slower  |  1.3 = 30% faster
# Applied via librosa time-stretch after generation.
SPEAKING_RATE = 1.0         # float  [0.5 – 2.0]

# ── PITCH SHIFT ─────────────────────────────────────────────────
# Semitones up (+) or down (-) from speaker's natural pitch.
PITCH_SHIFT = 0.0           # float  [-6.0 – +6.0]

# ── ENERGY / LOUDNESS ───────────────────────────────────────────
ENERGY_SCALE = 1.0          # float  [0.5 – 1.5]

# ── OUTPUT SAMPLE RATE ──────────────────────────────────────────
# Magpie is trained at 22050 Hz — this is the recommended value.
TARGET_SAMPLE_RATE = 22050  # int

# ── OUTPUT FILENAME ─────────────────────────────────────────────
OUTPUT_FILENAME = "magpie_tts_output.wav"

# ── TEXT CHUNK SIZE ─────────────────────────────────────────────
MAX_CHUNK_CHARS = 300       # int  [100 – 500]

# ── DEVICE ──────────────────────────────────────────────────────
import torch as _t
DEVICE = "cuda" if _t.cuda.is_available() else "cpu"

# ╔══════════════════════════════════════════════════════════════╗
# ║                    END OF CONFIGURATION                     ║
# ╚══════════════════════════════════════════════════════════════╝
print("✅ Configuration:")
print(f"   🎤 Speaker        : {SPEAKER_NAME.get(SPEAKER_ID, '?')} (ID {SPEAKER_ID})")
print(f"   🌐 Language       : {LANGUAGE}")
print(f"   ⚡ Speaking Rate  : {SPEAKING_RATE}×")
print(f"   🎵 Pitch Shift    : {PITCH_SHIFT:+.1f} semitones")
print(f"   🔊 Energy Scale   : {ENERGY_SCALE}")
print(f"   📻 Sample Rate    : {TARGET_SAMPLE_RATE} Hz")
print(f"   💻 Device         : {DEVICE.upper()}")
print(f"   📝 Chunk Size     : {MAX_CHUNK_CHARS} chars")
print(f"   💾 Output File    : {OUTPUT_FILENAME}")

---
## 🤖 Step 4 — Load the MagpieTTS Model
Downloads and loads the model using the official NeMo API.  
> ⏱️ First run downloads ~1.5 GB. Subsequent runs use the cached model.

In [ ]:
# ── PRE-PATCH FOR PYTHON 3.12 COMPATIBILITY ──
# Colab uses Python 3.12 which removed pkgutil.ImpImporter and FileFinder.find_module.
# Older deps like jieba and hydra-core crash without these monkey-patches.
import sys, pkgutil, importlib.machinery
if sys.version_info >= (3, 12):
    if not hasattr(pkgutil, "ImpImporter"):
        class _DummyImpImporter:
            def __init__(self, path=None): self.path = path
            def find_module(self, fullname, path=None): return None
            def iter_modules(self, prefix=""): return iter([])
        pkgutil.ImpImporter = _DummyImpImporter
    if not hasattr(importlib.machinery.FileFinder, "find_module"):
        def _dummy_find_module(self, fullname, path=None):
            spec = self.find_spec(fullname)
            return spec.loader if spec else None
        importlib.machinery.FileFinder.find_module = _dummy_find_module
# ─────────────────────────────────────────────

import torch, time

print("🔄 Loading MagpieTTS model (this may take 1–3 minutes on first run)...")
t0 = time.time()

from nemo.collections.tts.models import MagpieTTSModel

tts_model = MagpieTTSModel.from_pretrained("nvidia/magpie_tts_multilingual_357m")
tts_model = tts_model.eval()

if DEVICE == "cuda" and torch.cuda.is_available():
    tts_model = tts_model.cuda()

elapsed = time.time() - t0

print(f"\n{'━'*50}")
print(f"✅ Model loaded in {elapsed:.0f}s")
print(f"   Class  : {type(tts_model).__name__}")
print(f"   Device : {DEVICE.upper()}")
if DEVICE == "cuda":
    print(f"   GPU mem: {torch.cuda.memory_allocated()/1e6:.0f} MB")
print(f"{'━'*50}")

---
## 🎙️ Step 5 — Generate Speech

In [ ]:
import re, numpy as np, soundfile as sf, torch
from IPython.display import Audio, display

OUTPUT_PATH = f"/content/{OUTPUT_FILENAME}"


def split_text(text, max_chars=300):
    """Split at sentence boundaries; never cut mid-word."""
    sentences = re.split(r'(?<=[.!?…।])\s+', text.strip())
    chunks, cur = [], ""
    for s in sentences:
        candidate = (cur + " " + s).strip()
        if len(candidate) <= max_chars:
            cur = candidate
        else:
            if cur: chunks.append(cur)
            cur = s[:max_chars] if len(s) > max_chars else s
    if cur: chunks.append(cur)
    return [c for c in chunks if c.strip()]


def post_process(audio, sr):
    """Librosa speed/pitch/energy adjustments (all optional)."""
    audio = audio.astype(np.float32)
    try:
        import librosa
        if SPEAKING_RATE != 1.0:
            audio = librosa.effects.time_stretch(audio, rate=float(SPEAKING_RATE))
        if PITCH_SHIFT != 0.0:
            audio = librosa.effects.pitch_shift(audio, sr=sr, n_steps=float(PITCH_SHIFT))
    except Exception as e:
        print(f"      (post-process skipped: {e})")
    return np.clip(audio * float(ENERGY_SCALE), -1.0, 1.0).astype(np.float32)


# ── Main loop ────────────────────────────────────────────────────
print("🎙️  Generating speech...")
chunks = split_text(INPUT_TEXT, MAX_CHUNK_CHARS)
print(f"   Text   : {len(INPUT_TEXT):,} chars → {len(chunks)} chunk(s)\n")

audio_parts = []
current_sr  = TARGET_SAMPLE_RATE
silence     = np.zeros(int(current_sr * 0.256), dtype=np.float32)

for i, chunk in enumerate(chunks):
    preview = chunk[:72] + ("…" if len(chunk) > 72 else "")
    print(f"   [{i+1}/{len(chunks)}] {preview}")

    with torch.no_grad():
        audio, audio_len = tts_model.do_tts(
            chunk,
            language=LANGUAGE,
            apply_TN=APPLY_TEXT_NORM,
            speaker_index=SPEAKER_ID,
        )

    # audio is a torch tensor [1, T] or [T]
    if isinstance(audio, torch.Tensor):
        wav = audio.squeeze().cpu().float().numpy()
    else:
        wav = np.array(audio, dtype=np.float32).squeeze()

    # Trim to actual audio length
    if isinstance(audio_len, torch.Tensor):
        actual_len = audio_len.item()
    elif isinstance(audio_len, (int, float)):
        actual_len = int(audio_len)
    else:
        actual_len = len(wav)
    wav = wav[:actual_len]

    if len(wav) == 0:
        print(f"          ⚠️  Empty audio for chunk {i+1}, skipping")
        continue

    if not np.isfinite(wav).all():
        wav = np.nan_to_num(wav, nan=0.0, posinf=1.0, neginf=-1.0)

    wav = post_process(wav, current_sr)
    audio_parts.append(wav)
    if i < len(chunks) - 1:
        audio_parts.append(silence.copy())
    print(f"          ✓ {len(wav)/current_sr:.2f}s")

if not audio_parts:
    raise RuntimeError("❌ No audio generated — check errors above.")

final   = np.concatenate(audio_parts).astype(np.float32)
dur_tot = len(final) / current_sr

print(f"\n✅ Generation complete!")
print(f"   Duration    : {dur_tot:.2f}s ({dur_tot/60:.1f} min)")
print(f"   Sample rate : {current_sr} Hz")

sf.write(OUTPUT_PATH, final, current_sr, subtype="PCM_16")
print(f"   Saved to    : {OUTPUT_PATH}")
print("\n🔊 Preview:")
display(Audio(final, rate=current_sr))

---
## 📥 Step 6 — Download the Audio File

In [ ]:
import os
from google.colab import files as colab_files

OUTPUT_PATH = f"/content/{OUTPUT_FILENAME}"

if not os.path.exists(OUTPUT_PATH):
    raise FileNotFoundError(f"❌ {OUTPUT_PATH} not found — run Step 5 first.")

size_kb = os.path.getsize(OUTPUT_PATH) / 1024
print(f"✅  {OUTPUT_FILENAME}  ({size_kb:.1f} KB)  ready to download.")
print("\n⬇️  Downloading...")
colab_files.download(OUTPUT_PATH)
print("\n🎉 Done! Check your browser Downloads folder.")
print("   Tip: if download didn't start →  left sidebar → Files → right-click → Download")

---
## 🎁 Bonus — Try Different Voices
Run this cell to see available speakers.  
Then change `SPEAKER_ID` in Step 3 and re-run **Steps 3 → 5 → 6** (model stays loaded).

In [ ]:
print("🎤 Available Speakers:")
print()
speaker_map = {0: "John", 1: "Sofia", 2: "Aria", 3: "Jason", 4: "Leo"}
for sid, name in speaker_map.items():
    marker = " ◀── CURRENT" if sid == SPEAKER_ID else ""
    print(f"   {sid}: {name}{marker}")
print()
print("💡 Change SPEAKER_ID in Step 3 → re-run Step 3 → 5 → 6")